# Healthcare Analysis

## Objectives

* Check the contents of the raw Healthcare dataset
* Explore the data to check data types, summary statistics and basic patterns in the dataset
* Clean the data by handling missing data, duplicates and drop any unnecessary columns
* Check for any outliers
* De-identify the dataset by removing direct patient identifiers
* Feature engineer any columns that are required for EDA
* Save the cleaned data for Hypotheses testing and data visualising

## Inputs

* Kaggle Dataset: https://www.kaggle.com/datasets/prasad22/healthcare-dataset
* Python libraries: Pandas, Numpy and Matplotlib

## Outputs

* Save the cleaned data to the Dataset/CleanData folder as healthcare_cleaned_deidentified.csv 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/Healthcare_Analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/Healthcare_Analysis'

### Import Libraries 

In [4]:
import pandas as pd
import numpy as np

### Load the raw dataset

In [5]:
df = pd.read_csv('Dataset/RawData/healthcare_dataset.csv')
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


### Initial Inspection

I will look at the shape of the dataset using .shape

In [6]:
df.shape
print('This dataset has {} rows and {} columns'.format(df.shape[0], df.shape[1]))

This dataset has 55500 rows and 15 columns


.info() will give me a summary of the dataset which will show if there are any null values in each column as well as what datatype it is.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

This dataset has no missing values in any columns and states there is 1 float column, 2 integar columns and 12 object columns

.columns give me all the columns in the dataset and i have used .tolist() so it displays it as a list. This will help me to see if there are any columns I can remove that wont be useful for my analysis

In [8]:
df.columns.tolist()

['Name',
 'Age',
 'Gender',
 'Blood Type',
 'Medical Condition',
 'Date of Admission',
 'Doctor',
 'Hospital',
 'Insurance Provider',
 'Billing Amount',
 'Room Number',
 'Admission Type',
 'Discharge Date',
 'Medication',
 'Test Results']

Looking at the columns Name and Room Number stand out as ones I'll likely need to handle differently, Name is a direct patient identifier which I'll need to remove for privacy reasons and Room Number is just a facilities identifier with no analytical value. The rest look like they'll all be useful for my analysis.

.describe() provides me with a statistical summary of all the numeric columns, it shows the: Count, Mean, Std, Min, 25%, 50%, 75% and Max.
This helps me spot any obvious data quality issues such as outliers and negative values where there should'nt be. Ive rounded it to 2 decimal places to make it easier to read

In [9]:
df.describe().round(2)

,Age,Billing Amount,Room Number
count,55500.00,55500.00,55500.00
mean,51.54,25539.32,301.13
std,19.60,14211.45,115.24
min,13.00,-2008.49,101.00
25%,35.00,13241.22,202.00
50%,52.00,25538.07,302.00
75%,68.00,37820.51,401.00
max,89.00,52764.28,500.00


Age ranges from 13 to 89 with a mean of around 51.5, a wide but sensible spread. Billing Amount has a mean of around $25,539, but the minimum value is negative (-$2,008) which isn't possible for a real billing amount and is something I'll need to fix in the cleaning stage. Room Number just looks like a plain identifier field with no real analytical value.

.describe(include='object') provides me with a statistical summary of all the categorical (text-based) columns it shows the Count, Unique, Top and Freq.
This helps me check that each category has a sensible number of unique values and spot any inconsistent labelling, such as duplicate categories caused by different spelling or casing.

In [10]:
df.describe(include='object')

,Name,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Admission Type,Discharge Date,Medication,Test Results
count,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500
unique,49992,2,8,6,1827,40341,39876,5,3,1856,5,3
top,DAvId muNoZ,Male,A-,Arthritis,2024-03-16,Michael Smith,LLC Smith,Cigna,Elective,2020-03-15,Lipitor,Abnormal
freq,3,27774,6969,9308,50,27,44,11249,18655,53,11140,18627


Gender has 2 unique values, Blood Type has 8, Medical Condition has 6, Insurance Provider has 5, Admission Type has 3, Medication has 5, and Test Results has 3. Low counts with no obvious spelling or casing inconsistencies. Name has 49,992 unique values across 55,500 rows, showing a small number of repeated names(meaning i will need to remove them). Doctor and Hospital have a much larger number of unique values (40,341 and 39,876), which makes sense given how many different doctors and hospitals a dataset like this would realistically contain.

## Data Cleaning

### Missing values

To check missing values I will use .isnull().sum. This will show me if there any missing values in any columns

In [11]:
df.isnull().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

There are no missing values in any of the columns which is expected for a synthetically generated dataset a real hospital dataset would almost certainly have some missing values.

### Duplicates

To check for duplicate rows I will use .duplicated().sum(). This will show me how many fully duplicated rows exist in the dataset

In [13]:
df.duplicated().sum()

534

There are 534 fully duplicated rows in the dataset. Since these are exact duplicates across every column, including Name, Date of Admission and Doctor, they are almost certainly a generation artefact rather than genuine repeat admissions so I will drop them.

Before dropping them I want to take a quick look at a few of the duplicated rows to confirm they really are exact duplicates and not just similar looking records

In [14]:
df[df.duplicated(keep=False)].sort_values('Name').head(6)

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
54285,ABIgaIL YOung,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
42407,ABIgaIL YOung,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
26025,ALIcia taYLoR,78,Male,O+,Asthma,2022-09-18,Dawn Burton,Wright LLC,Aetna,31465.274979,149,Elective,2022-10-15,Aspirin,Inconclusive
53104,ALIcia taYLoR,78,Male,O+,Asthma,2022-09-18,Dawn Burton,Wright LLC,Aetna,31465.274979,149,Elective,2022-10-15,Aspirin,Inconclusive
50151,AMy GREEN,79,Female,B+,Obesity,2021-03-30,Brett Johnson,Taylor-Williamson,UnitedHealthcare,23402.358491,249,Elective,2021-04-27,Penicillin,Abnormal
42323,AMy GREEN,79,Female,B+,Obesity,2021-03-30,Brett Johnson,Taylor-Williamson,UnitedHealthcare,23402.358491,249,Elective,2021-04-27,Penicillin,Abnormal


In [15]:
# Now I will drop the duplicate rows and reset the index
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(54966, 15)

The dataset now has 54,966 rows after removing the 534 duplicates.

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [12]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)